# Day 2 Runner — Data Pipeline + Continued Pretraining (CPT)

Turn **Qwen3-1.7B-Base** into a domain-adapted base (`cpt-v1`) for an LLM-tutor.
Full fine-tuning on an A40 (~44 GB). Runs on a rented GPU (RunPod, etc.).

Pipeline: **collect → clean → split → tokenize/pack → CPT → perplexity.**
All settings live in `configs/day2.yaml`. Edit there, not in the cells.

> **Tested-on-RunPod notes are baked in:** the install cell handles the blinker /
> transformers / torchaudio issues, and every script call uses `{sys.executable}`
> so it runs under the same Python as this kernel.

## 0. GPU + code

In [ ]:
!nvidia-smi

In [ ]:
# Get the repo — pick ONE.
# Option A: clone
# !git clone https://github.com/vinmlops/llm-from-base-to-assistant.git
# %cd llm-from-base-to-assistant
# Option B: unzip an uploaded archive
# !unzip -q llm-from-base-to-assistant.zip && %cd llm-from-base-to-assistant
import os; print(os.getcwd())

## 1. Install dependencies (RunPod-safe)

Handles the three issues seen in practice:
- `--ignore-installed blinker` — system blinker has no pip record.
- force a Qwen3-capable **transformers** (and keep it on 4.x — 5.x changed the
  `TrainingArguments` API).
- remove **torchaudio** (unused; its compiled lib can fail to load).

**After this cell, restart the kernel once, then continue from cell 2 (skip re-install).**

In [ ]:
!pip install -q -r requirements.txt --ignore-installed blinker
!pip install -q -U "transformers>=4.51,<5.0" "tokenizers>=0.21" "torch>=2.4.0"
!pip uninstall -y -q torchaudio || true
print("Install done. RESTART THE KERNEL now, then run the check below.")

In [ ]:
import sys, torch, transformers
print("python:", sys.executable)
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("transformers:", transformers.__version__, "(want 4.51-4.x)")

## 2. Collect raw text
Papers (`ai-arxiv`) + scraped HF/PyTorch docs + theory (HF blog, HF LLM Course,
d2l, arXiv surveys) + FineWeb-Edu replay → `data/raw/`.
Flags: `--skip-docs`, `--skip-theory` if a source is flaky.

In [ ]:
import sys
!{sys.executable} data/collect.py --config configs/day2.yaml

## 3. Clean + deduplicate
Then **read ~20 cleaned docs by hand** — catches more than any metric.

In [ ]:
import sys
!{sys.executable} data/clean.py --config configs/day2.yaml

In [ ]:
from itertools import islice
import json
with open("data/clean/domain_papers.jsonl") as f:
    for line in islice(f, 2):
        r = json.loads(line); print(r["source"], "|", r["text"][:400], "\n---")

## 4. Split (document-level) + leakage check
Look for "✓ no leakage found".

In [ ]:
import sys
!{sys.executable} data/split.py --config configs/day2.yaml

## 5. Tokenize + pack
1024-token blocks, 85/15 domain:replay mixture, token-budget capped.

In [ ]:
import sys
!{sys.executable} data/tokenize_pack.py --config configs/day2.yaml

## 6. Continued pretraining (full fine-tuning)
Watch the loss: smooth descent = healthy; spikes = LR too high; rising **val** loss
= overfitting/forgetting. Saves `artifacts/cpt-v1`. (~15-60 min on the A40.)

> `cpt.py` is version-robust — it adapts its TrainingArguments to your installed
> transformers, so the 4.x/5.x API differences won't crash it.

In [ ]:
import sys
!{sys.executable} training/cpt.py --config configs/day2.yaml

## 7. Perplexity — did CPT work?
**Want:** domain perplexity ↓, general perplexity ≈ flat.

In [ ]:
import sys
!{sys.executable} evaluation/perplexity.py --config configs/day2.yaml

## 8. Preserve `cpt-v1` before teardown (Hugging Face Hub)

The rented machine's disk is temporary. Push the trained model to the Hub so it
survives. Needs a **Write** token from huggingface.co → Settings → Access Tokens.

In [ ]:
import os
print(os.listdir("artifacts/cpt-v1"))   # confirm the model saved

In [ ]:
from huggingface_hub import HfApi, login
login()  # paste your WRITE token when prompted
api = HfApi()
api.create_repo("vinmlops/cpt-v1", private=True, exist_ok=True)
api.upload_folder(folder_path="artifacts/cpt-v1", repo_id="vinmlops/cpt-v1")
print("Uploaded → https://huggingface.co/vinmlops/cpt-v1")

## ✅ Day 2 complete
You have `cpt-v1` (domain perplexity ↓, general flat), a versioned dataset
(`data/manifest.json`), and the model backed up on the Hub.

**Push CODE (not weights) to GitHub:**
```
!git add configs/ data/*.py training/ evaluation/ docs/ data/manifest.json requirements.txt
!git commit -m "Day 2: CPT pipeline + cpt-v1 lineage"
!git push
```

**Next — Day 3 (SFT):** generate instruction Q&A from this corpus, train with
assistant-only loss, and watch the model start to chat.